# Silver — physical_lojas

**Regras técnicas aplicadas nesta camada (planilha "Squad 3 —
Negócio/Técnica", Luiz Henrique):**
1. `id_loja` não pode ser nulo nem duplicado (PK).
2. `cnpj` deve ter 14 dígitos numéricos e ser único.
3. `estado_loja` deve ser UF válida (2 letras).

**Princípio geral desta camada:** a Silver **qualifica** os dados —
ela não decide o que será excluído dos KPIs. Registros com problema
ganham uma flag (`flag_<regra>_invalido`) e seguem no fluxo, exceto
quando o problema é uma violação de PK (nula ou duplicada), caso em
que o registro é movido para quarentena. A decisão de excluir ou
não um registro flagado é da **Gold**, de acordo com o propósito de
cada KPI.

**O que este notebook faz:**
- Lê o último batch da Bronze (`squad3/bronze/physical_lojas`).
- Usa as regras de qualidade versionadas de
  `governanca/00_data_quality_rules` (carregado transitivamente via
  `00_utils`) — fonte única de verdade.
- Aplica cast de tipos reais (a Bronze é toda string).
- Trata PK (`id_loja`): quarentena para nula/duplicada.
- Trata `cnpj`: valida 14 dígitos, preserva original, marca flag.
- Trata `estado_loja`: usa `correct_uf()` da governança, marca flag.
- Trata `nome_loja`/`cidade_loja` vazios/nulos.
- Registra métricas de qualidade por regra (para os gráficos do
  notebook `analysis`).
- Grava em Delta (`squad3/silver/physical_lojas`).
- **Não grava nada no SQL Server.**

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

# Modo de escrita da Silver:
#   "overwrite" -> reprocessamento completo (ambiente sujo, mudança de regras)
#   "append"    -> carga incremental (novos dados chegando no raw)
# Altere para "append" em execuções normais de atualização de dados.
SILVER_WRITE_MODE = "overwrite"

In [0]:
from pyspark.sql.functions import regexp_replace, length, trim, lower, udf
from pyspark.sql.types import StringType

## Leitura da Bronze

In [0]:
df_bronze = read_delta(BRONZE_LOJAS_PATH, adls_options)

print(f"Registros lidos da Bronze: {df_bronze.count()}")
display(df_bronze.limit(10))

## Verificação de pré-condição

In [0]:
verificar_destino_limpo(SILVER_LOJAS_PATH, adls_options, permitir_existente=True)

## Cast de tipos reais

A Bronze chega inteiramente como `string`. Aqui aplicamos os tipos
corretos antes de validar as regras técnicas.

In [0]:
from pyspark.sql.functions import expr

df_tipado = (
    df_bronze
    .withColumn("id_loja",     expr("TRY_CAST(id_loja AS BIGINT)"))
    .withColumn("cnpj",        col("cnpj"))
    .withColumn("nome_loja",   col("nome_loja"))
    .withColumn("cidade_loja", col("cidade_loja"))
    .withColumn("estado_loja", col("estado_loja"))
    .withColumn("peso_vendas", expr("TRY_CAST(peso_vendas AS DOUBLE)"))
)

## Regra técnica 1 — `id_loja` (PK não nula, não duplicada)

Registros com PK nula ou duplicada são movidos para a tabela de
quarentena (`squad3/silver/_quarentena_physical_lojas`). Apenas os
registros válidos seguem no fluxo principal.

In [0]:
total_antes_pk = df_tipado.count()

df_pk_valida = separar_quarentena_pk(
    df=df_tipado,
    coluna_pk="id_loja",
    quarentena_path=SILVER_QUARENTENA_LOJAS_PATH,
    adls_options=adls_options,
)

qtd_quarentenados_pk = total_antes_pk - df_pk_valida.count()

registrar_metrica_dq(
    tabela="physical_lojas",
    regra="01_pk_id_loja_nula_ou_duplicada",
    qtd_registros_afetados=qtd_quarentenados_pk,
    qtd_registros_total=total_antes_pk,
    adls_options=adls_options,
)

print(f"[REGRA 1] {qtd_quarentenados_pk} registro(s) movido(s) para quarentena (PK).")


## Regra técnica 2 — `cnpj` (14 dígitos numéricos, único)

O valor original é sempre preservado em `cnpj`. Quando inválido, o
campo `cnpj_tratado` recebe `"NAO_INFORMADO"` e `flag_cnpj_invalido`
é marcada como `true` — a Gold decide se um CNPJ inválido deve ou
não excluir a loja de um KPI específico.

In [0]:
df_cnpj = (
    df_pk_valida
    .withColumn("cnpj_limpo", regexp_replace(col("cnpj"), r"[^0-9]", ""))
    .withColumn(
        "flag_cnpj_invalido",
        (col("cnpj_limpo").isNull())
        | (length(col("cnpj_limpo")) != 14)
    )
    .withColumn(
        "cnpj_tratado",
        when(col("flag_cnpj_invalido"), lit("NAO_INFORMADO")).otherwise(col("cnpj_limpo"))
    )
    .drop("cnpj_limpo")
)

qtd_cnpj_invalido = df_cnpj.filter(col("flag_cnpj_invalido")).count()
total_cnpj = df_cnpj.count()

registrar_metrica_dq(
    tabela="physical_lojas",
    regra="02_cnpj_invalido",
    qtd_registros_afetados=qtd_cnpj_invalido,
    qtd_registros_total=total_cnpj,
    adls_options=adls_options,
)

print(f"[REGRA 2] {qtd_cnpj_invalido} de {total_cnpj} registro(s) com CNPJ inválido.")


####CNPJ duplicado entre lojas

A regra anterior só validava o formato do CNPJ (14 dígitos), não se o mesmo CNPJ aparecia em mais de uma loja distinta. Duas lojas com o mesmo CNPJ é um problema de integridade cadastral que hoje passa silenciosamente. A flag flag_cnpj_duplicado marca todas as ocorrências de um CNPJ válido que se repete em mais de um id_loja.

In [0]:
from pyspark.sql.functions import count as spark_count
from pyspark.sql.window import Window

janela_cnpj = Window.partitionBy("cnpj_tratado")

df_cnpj = (
    df_cnpj
    .withColumn(
        "_qtd_lojas_com_mesmo_cnpj",
        spark_count("id_loja").over(janela_cnpj)
    )
    .withColumn(
        "flag_cnpj_duplicado",
        (col("cnpj_tratado") != "NAO_INFORMADO") & (col("_qtd_lojas_com_mesmo_cnpj") > 1)
    )
    .drop("_qtd_lojas_com_mesmo_cnpj")
)

qtd_cnpj_duplicado = df_cnpj.filter(col("flag_cnpj_duplicado")).count()
total_cnpj_dup = df_cnpj.count()

registrar_metrica_dq(
    tabela="physical_lojas",
    regra="02b_cnpj_duplicado_entre_lojas",
    qtd_registros_afetados=qtd_cnpj_duplicado,
    qtd_registros_total=total_cnpj_dup,
    adls_options=adls_options,
)

print(f"[REGRA 2b] {qtd_cnpj_duplicado} de {total_cnpj_dup} registro(s) "
      f"com CNPJ duplicado entre lojas diferentes.")
if qtd_cnpj_duplicado > 0:
    display(
        df_cnpj.filter(col("flag_cnpj_duplicado"))
        .select("id_loja", "nome_loja", "cnpj_tratado")
        .orderBy("cnpj_tratado")
    )

## Regra técnica 3 — `estado_loja` (UF válida)

Usa `correct_uf()` (definida em `governanca/00_data_quality_rules`,
fonte única de verdade do dicionário de correção de UF). O valor
original é preservado em `estado_loja`; o valor corrigido fica em
`estado_tratado`. Quando `correct_uf()` não consegue resolver,
retorna `"NAO_INFORMADO"` e a flag é marcada.

In [0]:
correct_uf_udf = udf(correct_uf, StringType())

df_uf = (
    df_cnpj
    .withColumn("estado_tratado", correct_uf_udf(col("estado_loja")))
    .withColumn("flag_uf_invalido", col("estado_tratado") == lit("NAO_INFORMADO"))
)

qtd_uf_invalido = df_uf.filter(col("flag_uf_invalido")).count()
total_uf = df_uf.count()

registrar_metrica_dq(
    tabela="physical_lojas",
    regra="03_estado_loja_uf_invalido",
    qtd_registros_afetados=qtd_uf_invalido,
    qtd_registros_total=total_uf,
    adls_options=adls_options,
)


print(f"[REGRA 3] {qtd_uf_invalido} de {total_uf} registro(s) com UF inválida.")

## Tratamentos complementares — `nome_loja` e `cidade_loja`

Sem flag de qualidade dedicada (não fazem parte das 3 regras
técnicas formais da planilha), mas precisam de um valor padrão
para evitar nulos/"nan" chegando à Gold.

In [0]:
df_completo = (
    df_uf
    .withColumn(
        "nome_loja",
        when(
            col("nome_loja").isNull() | (trim(col("nome_loja")) == "") | (lower(col("nome_loja")) == "nan"),
            lit("Nao Informado")
        ).otherwise(col("nome_loja"))
    )
    .withColumn(
        # Bug corrigido: a regra anterior só pegava nome_loja IGUAL a
        # "nan" inteiro -- não cobria o caso real encontrado na carga
        # nova: "nan" como SUFIXO dentro do nome (ex.: "Loja Itarana -
        # nan", padrão "Loja {cidade} - {bairro}" com bairro ausente).
        # regexp_replace remove esse sufixo " - nan" (case-insensitive,
        # com ou sem espaços) deixando só a parte válida do nome.
        "nome_loja",
        regexp_replace(
            col("nome_loja"),
            r"(?i)\s*-\s*nan\s*$",
            ""
        )
    )
    .withColumn(
        "cidade_loja",
        when(
            # Frente C: antes só cobria nulo/vazio — não pegava a string
            # literal "nan" (minúscula ou variações de caixa), que é
            # exatamente o que aparecia nos gráficos do Looker
            # ("Loja Itapecerica - nan", "Loja Bom Sucesso - nan").
            col("cidade_loja").isNull()
            | (trim(col("cidade_loja")) == "")
            | (lower(trim(col("cidade_loja"))) == "nan"),
            lit("NAO_INFORMADO")
        ).otherwise(col("cidade_loja"))
    )
)

## Metadados de auditoria + escrita em Delta


####Gap na sequência de id_loja

Checagem automática: compara o total de lojas com o maior id_loja existente. Se houver diferença, significa que existe pelo menos um id "faltando" na sequência — hoje isso é descoberto manualmente pelo cliente (ex.: falta o id 14, maior id é 30 para 29 lojas). Gera um alerta explícito no log em vez de ficar em silêncio.



In [0]:
qtd_lojas_total = df_completo.select("id_loja").distinct().count()
maior_id_loja = df_completo.agg({"id_loja": "max"}).collect()[0][0]

ids_existentes = set(
    row["id_loja"] for row in df_completo.select("id_loja").distinct().collect()
)
ids_esperados = set(range(1, maior_id_loja + 1))
ids_faltando = sorted(ids_esperados - ids_existentes)

if ids_faltando:
    print(
        f"[ALERTA REGRA 04] Gap na sequência de id_loja detectado: "
        f"{len(ids_faltando)} id(s) ausente(s) entre 1 e {maior_id_loja} "
        f"-> {ids_faltando}. Total de lojas ativas: {qtd_lojas_total}."
    )
else:
    print(
        f"[REGRA 04] Sequência de id_loja OK: {qtd_lojas_total} loja(s), "
        f"sem gaps até o id {maior_id_loja}."
    )

registrar_metrica_dq(
    tabela="physical_lojas",
    regra="04_gap_sequencia_id_loja",
    qtd_registros_afetados=len(ids_faltando),
    qtd_registros_total=maior_id_loja,
    adls_options=adls_options,
)

In [0]:
df_silver_final = adicionar_metadados_silver(df_completo)

(
write_delta(df_silver_final, SILVER_LOJAS_PATH, adls_options, mode=SILVER_WRITE_MODE)
)

print(f"[OK] {df_silver_final.count()} linha(s) gravada(s) em '{SILVER_LOJAS_PATH}'.")
display(df_silver_final.limit(10))
